
# STEP 1: Loading the API Key

In [9]:
from dotenv import load_dotenv
import os

load_dotenv("/content/.env")

API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")

print(f"✓ API Key loaded!")

✓ API Key loaded!


In [10]:
import requests
import pandas as pd
from datetime import datetime

In [11]:
try:
    from google.colab import userdata
    API_KEY = userdata.get('API_KEY')
except Exception:
    # Για local Python
    import os
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("⚠️ API_KEY not found! Check Colab Secrets or .env file")

print(f"✓ API Key loaded: {API_KEY[:10]}...***")

✓ API Key loaded: 55a954a67b...***


# STEP 2: Function to get the weather

In [12]:
def get_weather(city_name):
    """
    Gets weather data for a city with OpenWeatherMap
    """
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"  # Celsius
    }

    try:
        response = requests.get(url, params=params, timeout=5)

        if response.status_code == 200:
            return response.json()
        elif response.status_code == 401:
            print("❌ Error: Invalid API key!")
            return None
        elif response.status_code == 404:
            print(f"❌ Error: City '{city_name}' not found!")
            return None
        elif response.status_code == 429:
            print("❌ Error: You have exceeded your call limit (1000/day)")
            return None
        else:
            print(f"❌ Error: {response.status_code}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"❌ Connection error: {e}")
        return None

# STEP 3: Function to display data

In [13]:
def display_weather_detailed(city_name, data):
    """
    Displays weather data in structured format
    """
    if data is None:
        return

    print(f"\n{'='*60}")
    print(f"WEATHER IN {data['name'].upper()}, {data['sys']['country']}")
    print(f"{'='*60}\n")

    # Κύρια δεδομένα
    main = data['main']
    weather = data['weather'][0]
    wind = data['wind']
    clouds = data['clouds']

    print(f"🌡️  TEMPERATURE")
    print(f"   Current:      {main['temp']}°C")
    print(f"   Sense:       {main['feels_like']}°C")
    print(f"   Min:           {main['temp_min']}°C")
    print(f"   Max:           {main['temp_max']}°C")

    print(f"\n💨 WIND")
    print(f"   Speed:      {wind['speed']} m/s")
    if 'gust' in wind:
        print(f"   Gust:          {wind['gust']} m/s")

    print(f"\n💧 HUMIDITY & PRESSURE")
    print(f"   Humidity:       {main['humidity']}%")
    print(f"   Pressure:         {main['pressure']} hPa")

    print(f"\n☁️ CLOUD & DESCRIPTION")
    print(f"   Description:     {weather['description'].capitalize()}")
    print(f"   Clouds:   {clouds['all']}%")

    print(f"\n🌅 SUNRISE/SUNSET")
    sunrise = datetime.fromtimestamp(data['sys']['sunrise'])
    sunset = datetime.fromtimestamp(data['sys']['sunset'])
    print(f"   Sunrise:       {sunrise.strftime('%H:%M')}")
    print(f"   Sunset:          {sunset.strftime('%H:%M')}")

    print(f"\n📍 LOCATION")
    print(f"   Latitude:  {data['coord']['lat']}")
    print(f"   Longitude:   {data['coord']['lon']}")

# STEP 4: Run for multiple cities

In [14]:
cities = ["Athens", "Thessaloniki", "London", "New York", "Paris"]

print("\n🌍 WEATHER DATA RECOVERY...\n")

weather_data_list = []

for city in cities:
    weather_data = get_weather(city)

    if weather_data:
        display_weather_detailed(city, weather_data)

        # Save the data for later
        weather_data_list.append({
            "City": weather_data["name"],
            "Country": weather_data["sys"]["country"],
            "Temperature (°C)": weather_data["main"]["temp"],
            "Feels Like (°C)": weather_data["main"]["feels_like"],
            "Humidity (%)": weather_data["main"]["humidity"],
            "Pressure (hPa)": weather_data["main"]["pressure"],
            "Wind Speed (m/s)": weather_data["wind"]["speed"],
            "Description": weather_data["weather"][0]["description"],
            "Cloudiness (%)": weather_data["clouds"]["all"]
        })


🌍 WEATHER DATA RECOVERY...


WEATHER IN ATHENS, GR

🌡️  TEMPERATURE
   Current:      19.35°C
   Sense:       19.46°C
   Min:           18.49°C
   Max:           20.09°C

💨 WIND
   Speed:      5.36 m/s
   Gust:          7.6 m/s

💧 HUMIDITY & PRESSURE
   Humidity:       81%
   Pressure:         1016 hPa

☁️ CLOUD & DESCRIPTION
   Description:     Scattered clouds
   Clouds:   40%

🌅 SUNRISE/SUNSET
   Sunrise:       05:12
   Sunset:          15:10

📍 LOCATION
   Latitude:  37.9795
   Longitude:   23.7162

WEATHER IN THESSALONIKI, GR

🌡️  TEMPERATURE
   Current:      16.8°C
   Sense:       16.57°C
   Min:           15.95°C
   Max:           18.28°C

💨 WIND
   Speed:      4.12 m/s

💧 HUMIDITY & PRESSURE
   Humidity:       78%
   Pressure:         1013 hPa

☁️ CLOUD & DESCRIPTION
   Description:     Few clouds
   Clouds:   20%

🌅 SUNRISE/SUNSET
   Sunrise:       05:21
   Sunset:          15:06

📍 LOCATION
   Latitude:  40.6403
   Longitude:   22.9439

WEATHER IN LONDON, GB

🌡️  TEMPERATURE


# STEP 5: Create DataFrame and CSV

In [15]:
if weather_data_list:
    df = pd.DataFrame(weather_data_list)

    print(f"\n\n{'='*60}")
    print("📊 DATA SUMMARY")
    print(f"{'='*60}\n")
    print(df.to_string(index=False))

    # Αποθήκευση σε CSV
    filename = "weather_data_openweathermap.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n✓ Data stored in {filename}")

    # Στατιστικά
    print(f"\n📈 STATISTICS")
    print(f"   Average temperature:  {df['Temperature (°C)'].mean():.1f}°C")
    print(f"   Maximum:           {df['Temperature (°C)'].max():.1f}°C")
    print(f"   Minimum:          {df['Temperature (°C)'].min():.1f}°C")
else:
    print("\n❌ Data could not be retrieved")



📊 DATA SUMMARY

        City Country  Temperature (°C)  Feels Like (°C)  Humidity (%)  Pressure (hPa)  Wind Speed (m/s)      Description  Cloudiness (%)
      Athens      GR             19.35            19.46            81            1016              5.36 scattered clouds              40
Thessaloniki      GR             16.80            16.57            78            1013              4.12       few clouds              20
      London      GB              1.30            -2.41            86            1020              3.60        clear sky               2
    New York      US              5.77             4.21            64            1022              2.06        clear sky               0
       Paris      FR             -0.47            -4.57            89            1018              3.60 scattered clouds              40

✓ Data stored in weather_data_openweathermap.csv

📈 STATISTICS
   Average temperature:  8.6°C
   Maximum:           19.4°C
   Minimum:          -0.5°C
